In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [3]:
docs = [
    "Machine learning is a field of artificial intelligence that allows computers to learn from data.",
    "Deep learning uses neural networks with many layers to learn complex patterns.",
    "A vector database stores embeddings and supports similarity search.",
    "Natural language processing helps computers understand and generate human language.",
    "Recommendation systems suggest items to users based on their behavior and preferences.",
    "Fraud detection systems identify suspicious transactions or activities.",
    "Clustering is an unsupervised learning technique used to group similar data points.",
    "Classification is a supervised learning task where the model predicts a category.",
    "Regression is used to predict continuous numerical values.",
    "Feature engineering is the process of creating useful input variables for machine learning models."
]

docs2 = [
    "Cơ sở dữ liệu vector được dùng để lưu trữ embedding và tìm kiếm theo độ tương đồng.",
    "Embedding là cách biểu diễn văn bản dưới dạng vector số.",
    "Cosine similarity được dùng để đo mức độ giống nhau giữa hai vector.",
    "RAG là kỹ thuật kết hợp truy xuất tài liệu và sinh câu trả lời bằng mô hình ngôn ngữ.",
    "FAISS là thư viện hỗ trợ tìm kiếm vector nhanh.",
    "ChromaDB thường được dùng để xây dựng chatbot hỏi đáp tài liệu trên máy local.",
    "Qdrant là vector database hỗ trợ tìm kiếm theo vector và lọc metadata.",
    "SentenceTransformer có thể biến câu hoặc đoạn văn thành embedding.",
    "Recommendation system dùng dữ liệu hành vi người dùng để gợi ý sản phẩm.",
    "Fraud detection giúp phát hiện giao dịch bất thường hoặc gian lận."
]

In [5]:
model = SentenceTransformer("sentence-transformers/multi-qa-mpnet-base-dot-v1")

docs_embedding = model.encode(docs)

In [9]:
print(docs_embedding)
print(docs_embedding.shape)

[[-0.05358981  0.14974287 -0.26576433 ... -0.21625586  0.24924955
  -0.293759  ]
 [-0.0584028  -0.2883316  -0.33590928 ... -0.32224664 -0.06296873
  -0.38015872]
 [ 0.00642635 -0.39691654 -0.27812248 ... -0.01470365  0.03039642
  -0.21502326]
 ...
 [-0.03284151  0.03602678 -0.19921279 ... -0.18379305  0.07236396
  -0.23236443]
 [-0.254045   -0.25764155 -0.30564582 ...  0.08737081 -0.21833362
  -0.18890598]
 [ 0.27796686 -0.3084479  -0.25079194 ...  0.07142209 -0.12079901
  -0.3912999 ]]
(10, 768)


In [12]:
query = 'What is Recommendation systems?'
query_embedding = model.encode(query)
print(query_embedding)

[ 9.98161286e-02 -4.48631614e-01 -4.36138272e-01 -3.86734456e-02
  1.38950080e-01 -1.66161269e-01  1.05605945e-01 -3.51259671e-02
  1.54892012e-01  3.19785252e-02 -2.23153815e-01  1.19058073e-01
 -1.28917277e-01  1.17466394e-02 -1.77303091e-01  1.14486463e-01
  9.70340148e-02  5.95601164e-02 -1.01867363e-01 -1.95552930e-01
  7.80437887e-02  3.44355218e-03  4.96040136e-02  5.66491485e-02
 -7.32163712e-02 -1.50925279e-01 -1.59335986e-01 -1.34875238e-01
 -3.74098539e-01 -5.97561672e-02 -9.49914753e-02  4.24386114e-02
 -1.54294044e-01  2.59916961e-01 -9.25505010e-05 -2.31066957e-01
  9.27751362e-02  7.53955990e-02 -2.68998444e-01  1.06466636e-01
 -3.46818954e-01 -5.73074967e-02  2.61401892e-01 -2.29392111e-01
 -8.83662701e-02 -3.95638943e-02  1.38522699e-01 -1.91422179e-01
  2.73664623e-01 -2.59908512e-02  4.43164885e-01  1.06278434e-01
 -1.98982641e-01 -1.64592609e-01 -1.32951736e-02  1.95983320e-01
 -2.53288209e-01  1.24664709e-01  4.67565596e-01  3.60493828e-03
  1.60390988e-01 -2.24879

In [15]:
similarity = model.similarity(query_embedding, docs_embedding)
print(similarity)

tensor([[14.4052, 14.8824, 17.7524, 12.7860, 26.0329, 16.3192, 15.3560, 16.3683,
         13.8041, 11.7450]])


## FAISS - Facebook AI Similarity Search

![faiss](https://media.geeksforgeeks.org/wp-content/uploads/20260109180837246689/search_query.webp)

In [19]:
dimension = docs_embedding.shape[1]
print("Dim:", dimension)

Dim: 768


In [21]:
# Create index
index = faiss.IndexFlatL2(dimension)
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001867BA4FDE0> >

In [24]:
index.ntotal

0

In [39]:
docs_embeddings = np.array(docs_embedding).astype("float32")
index.add(docs_embeddings)
index.ntotal

30

In [45]:
query_embedding.shape
query_embedding = query_embedding.reshape(1, -1)
query_embedding = np.array(query_embedding).astype("float32")

In [62]:
# remove index 10-20
index.remove_ids(np.arange(10, 20))

10

In [63]:
index.ntotal

10

In [64]:
k = 3
distances, indices = index.search(query_embedding, k)
print(indices)
for idx, distance in zip(indices[0], distances[0]):
    print(docs[idx], distance)

[[4 7 5]]
Recommendation systems suggest items to users based on their behavior and preferences. 14.253436
Classification is a supervised learning task where the model predicts a category. 34.282684
Fraud detection systems identify suspicious transactions or activities. 34.666615
